# Evaluation Results

This notebook evaluates baseline salary models using the processed train/test files created by `train_test_processed.ipynb`.

It does not rebuild raw features. The processed files already enforce the intended split:

- Train: 2021-2024
- Test: 2025 holdout
- PCA and preprocessing fit on train only, then applied to 2025


## 1. Load Processed Files


In [29]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score, root_mean_squared_error

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

processed_dir = repo / "data" / "processed"
results_dir = repo / "results"
results_dir.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(processed_dir / "X_train_processed.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")["salary"]
X_test = pd.read_csv(processed_dir / "X_test_2025_processed.csv")
y_test = pd.read_csv(processed_dir / "y_test_2025.csv")["salary"]
lookup_train = pd.read_csv(processed_dir / "player_lookup_train.csv")
lookup_test = pd.read_csv(processed_dir / "player_lookup_test_2025.csv")
feature_names = pd.read_csv(processed_dir / "feature_names.csv")["feature"].tolist()

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}, lookup_train: {lookup_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}, lookup_test:  {lookup_test.shape}")


X_train: (742, 22), y_train: (742,), lookup_train: (742, 5)
X_test:  (223, 22), y_test:  (223,), lookup_test:  (223, 5)


## 2. Validate Evaluation Inputs

These checks verify row alignment, split years, feature names, and missing/infinite values before model evaluation.


In [30]:
def is_clean(df):
    return bool(df.isna().any().any() or np.isinf(df.to_numpy()).any())

validation_rows = [
    {"check": "train row alignment", "passed": len(X_train) == len(y_train) == len(lookup_train), "detail": f"X={len(X_train)}, y={len(y_train)}, lookup={len(lookup_train)}"},
    {"check": "test row alignment", "passed": len(X_test) == len(y_test) == len(lookup_test), "detail": f"X={len(X_test)}, y={len(y_test)}, lookup={len(lookup_test)}"},
    {"check": "train years are 2021-2024", "passed": set(lookup_train["year"].unique()) == {2021, 2022, 2023, 2024}, "detail": sorted(lookup_train["year"].unique())},
    {"check": "test year is 2025", "passed": set(lookup_test["year"].unique()) == {2025}, "detail": sorted(lookup_test["year"].unique())},
    {"check": "no NaN or inf in X_train", "passed": not is_clean(X_train), "detail": f"missing={int(X_train.isna().sum().sum())}"},
    {"check": "no NaN or inf in X_test", "passed": not is_clean(X_test), "detail": f"missing={int(X_test.isna().sum().sum())}"},
    {"check": "no NaN or inf in y_train", "passed": not y_train.isna().any() and not np.isinf(y_train.to_numpy()).any(), "detail": f"missing={int(y_train.isna().sum())}"},
    {"check": "no NaN or inf in y_test", "passed": not y_test.isna().any() and not np.isinf(y_test.to_numpy()).any(), "detail": f"missing={int(y_test.isna().sum())}"},
    {"check": "feature names match X columns", "passed": feature_names == X_train.columns.tolist() == X_test.columns.tolist(), "detail": f"features={len(feature_names)}"},
]

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

,check,passed,detail
0,train row alignment,True,"X=742, y=742, lookup=742"
1,test row alignment,True,"X=223, y=223, lookup=223"
2,train years are 2021-2024,True,"[2021, 2022, 2023, 2024]"
3,test year is 2025,True,[2025]
4,no NaN or inf in X_train,True,missing=0
5,no NaN or inf in X_test,True,missing=0
6,no NaN or inf in y_train,True,missing=0
7,no NaN or inf in y_test,True,missing=0
8,feature names match X columns,True,features=22


## 3. Baseline Evaluation

Each baseline model is fit on 2021-2024 and evaluated once on the 2025 holdout set.

The same table also includes chronological CV scores within the training period:

- Fold 1: train 2021, validate 2022
- Fold 2: train 2021-2022, validate 2023
- Fold 3: train 2021-2023, validate 2024


In [33]:
# Define simple baseline models for evaluation
models = {
    "DummyRegressor": DummyRegressor(strategy="mean"),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=5,
        random_state=26,
    ),
}

# Record the split design used in the results file
split_strategy = "train(2021_2024)/test(2025)"
cv_strategy = "chronological_cross_validation"
generated_at = pd.Timestamp.now().isoformat(timespec="seconds")

metric_rows = []
prediction_frames = []

# Build chronological CV folds inside the training period only
train_years = sorted(lookup_train["year"].unique())
cv_folds = []
for fold, val_year in enumerate(train_years[1:], start=1):
    fold_train_years = [year for year in train_years if year < val_year]
    train_idx = lookup_train[lookup_train["year"].isin(fold_train_years)].index
    val_idx = lookup_train[lookup_train["year"] == val_year].index
    cv_folds.append((fold, fold_train_years, val_year, train_idx, val_idx))

for model_name, model in models.items():
    # First evaluate the model on chronological CV folds
    cv_rows = []
    for fold, fold_train_years, val_year, train_idx, val_idx in cv_folds:
        cv_model = model.__class__(**model.get_params())
        cv_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        cv_pred = cv_model.predict(X_train.iloc[val_idx])
        cv_y = y_train.iloc[val_idx]
        nonzero_cv = cv_y != 0

        cv_rows.append({
            "fold": fold,
            "train_years": ",".join(str(y) for y in fold_train_years),
            "validation_year": val_year,
            "rmse": root_mean_squared_error(cv_y, cv_pred),
            "mae": mean_absolute_error(cv_y, cv_pred),
            "mape": mean_absolute_percentage_error(cv_y[nonzero_cv], cv_pred[nonzero_cv]) if nonzero_cv.any() else np.nan,
            "r2": r2_score(cv_y, cv_pred),
            "n_train_fold": len(train_idx),
            "n_validation_fold": len(val_idx),
        })

    cv_df = pd.DataFrame(cv_rows)

    # Then fit on all 2021-2024 training data and evaluate once on 2025
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    # Store CV summary metrics and final 2025 holdout metrics in one row per model
    metric_rows.append({
        "model": model_name,
        "split_strategy": split_strategy,
        "cv_strategy": cv_strategy,
        "train_years": "2021-2024",
        "test_year": 2025,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "cv_rmse_mean": cv_df["rmse"].mean(),
        "cv_rmse_std": cv_df["rmse"].std(ddof=0),
        "cv_mae_mean": cv_df["mae"].mean(),
        "cv_mae_std": cv_df["mae"].std(ddof=0),
        "cv_mape_mean": cv_df["mape"].mean(),
        "cv_mape_std": cv_df["mape"].std(ddof=0),
        "cv_r2_mean": cv_df["r2"].mean(),
        "holdout_rmse": root_mean_squared_error(y_test, pred),
        "holdout_mae": mean_absolute_error(y_test, pred),
        "holdout_mape": mean_absolute_percentage_error(y_test, pred),
        "holdout_r2": r2_score(y_test, pred),
        "generated_at": generated_at,
    })

    # Save player-level 2025 predictions for later under/overvaluation analysis
    pred_df = lookup_test.copy()
    pred_df["test_row_id"] = pred_df.index
    pred_df["model"] = model_name
    pred_df["actual_salary"] = y_test.to_numpy()
    pred_df["predicted_salary"] = pred
    pred_df["residual"] = pred_df["actual_salary"] - pred_df["predicted_salary"]
    pred_df["value_gap_predicted_minus_actual"] = pred_df["predicted_salary"] - pred_df["actual_salary"]
    pred_df["absolute_error"] = pred_df["residual"].abs()
    pred_df["split_strategy"] = split_strategy
    pred_df["generated_at"] = generated_at
    prediction_frames.append(pred_df)

baseline_results = pd.DataFrame(metric_rows).sort_values("holdout_rmse")
player_predictions = pd.concat(prediction_frames, ignore_index=True)

display(baseline_results)


,model,split_strategy,cv_strategy,train_years,test_year,n_train,n_test,cv_rmse_mean,cv_rmse_std,cv_mae_mean,cv_mae_std,cv_mape_mean,cv_mape_std,cv_r2_mean,holdout_rmse,holdout_mae,holdout_mape,holdout_r2,generated_at
3,RandomForestRegressor,train(2021_2024)/test(2025),chronological_cross_validation,2021-2024,2025,742,223,39984.835672,3447.832852,29489.863345,4192.844763,1.637227,0.755579,0.655671,41615.807951,30503.017235,1.290567,0.629818,2026-06-17T20:25:37
2,Ridge,train(2021_2024)/test(2025),chronological_cross_validation,2021-2024,2025,742,223,39959.091551,2765.690610,29676.579243,3446.472105,1.675810,0.839704,0.657388,44797.771437,34805.999746,1.911271,0.571046,2026-06-17T20:25:37
1,LinearRegression,train(2021_2024)/test(2025),chronological_cross_validation,2021-2024,2025,742,223,40737.848988,2990.896306,30446.494996,3780.938186,1.736685,0.915375,0.643572,45158.132345,34968.851884,1.931045,0.564117,2026-06-17T20:25:37
0,DummyRegressor,train(2021_2024)/test(2025),chronological_cross_validation,2021-2024,2025,742,223,69031.711334,958.521324,55568.050056,922.502585,4.657354,0.972842,-0.015651,68539.787794,54790.731939,4.145445,-0.004116,2026-06-17T20:25:37


### MAPE Limitation

Despite removing observations with zero salary, MAPE still exceeds 100%. This is still because the dataset contains multiple contract structures with substantially different salary scales (e.g., rookie, hardship, and veteran contracts). As a result, even moderate dollar prediction errors for low-salary contracts can translate into very large percentage errors.

## 4. Save Results

The metrics table and player-level predictions are saved under `results/`.

Result Guideline :
- Intermediate CSV/JSON tables of cross-validation scores, metric breakdowns, calibration curves.
- Each file must specify the split strategy and date generated in its filename or metadata.

In [34]:
metrics_path = results_dir / "baseline_evaluation.csv"
predictions_path = results_dir / "2025_player_salary_predictions.csv"

baseline_results.to_csv(metrics_path, index=False)

# Save predictions as one row per 2025 test row, with one prediction column per non-dummy model
player_predictions = (
    player_predictions[player_predictions["model"] != "DummyRegressor"]
    .pivot_table(
        index=["test_row_id", "player", "team", "year", "group", "actual_salary"],
        columns="model",
        values="predicted_salary",
        aggfunc="first",
    )
    .reset_index()
)
player_predictions = player_predictions.rename(
    columns={col: f"{col}_predicted_salary" for col in player_predictions.columns if col not in ["test_row_id", "player", "team", "year", "group", "actual_salary"]}
)
player_predictions.to_csv(predictions_path, index=False)